In [48]:
#Import the required packages
#Import package pandas for data analysis
import pandas as pd


In [49]:
def build_dqr_table(df, columns):
    """
    Build and save a Data Quality Report table.
    
    Parameters:
    -----------
    df : pandas.DataFrame
    columns : Index
        Features to include
    csv_filename : str
        Output filename
    
    Returns:
    --------
    pandas.DataFrame
        Complete DQR table
    """
    stats = df[columns].describe().T
    perc_missing = 100 * (df[columns].isnull().sum() / len(df))
    cardinality = df[columns].nunique()
    
    dqr_table = pd.concat([
        stats,
        pd.DataFrame(perc_missing, columns=['%missing']),
        pd.DataFrame(cardinality, columns=['cardinality'])
    ], axis=1)
    
    return dqr_table


print("✓ Helper function loaded successfully")

✓ Helper function loaded successfully


In [50]:
def load_data(filename):
    # Reading from a csv file, into a data frame with error handling
    try:
        # If the csv file has any special symbols, it helps to specify the file encoding.
        df = pd.read_csv(filename, 
                        keep_default_na=True, 
                        delimiter=',', 
                        skipinitialspace=True)
        
        print(f"✓ Successfully loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns")
        
        # Show the first few rows in the data frame
        display(df.head(5))
        
    except FileNotFoundError:
        print("ERROR: The file 'ppr-group-25204989-train.csv' was not found.")
        print("Please ensure the file is in the current working directory.")
    except Exception as e:
        print(f"ERROR: An unexpected error occurred: {e}")

    return df

In [51]:
# Validate that the dataframe has data and expected columns
def validate_data(df):
    print("Dataset Validation Checks:")
    print("="*60)

    # Check 1: Dataset is not empty
    if df.empty:
        print("⚠ WARNING: Dataset is empty!")
    else:
        print(f"✓ Dataset contains {df.shape[0]:,} rows")

    # Check 2: Check for duplicate rows
    duplicate_count = df.duplicated().sum()
    if duplicate_count > 0:
        print(f"⚠ WARNING: Found {duplicate_count} duplicate rows")
    else:
        print("✓ No duplicate rows found")

    # Check 3: Check if all columns are unnamed
    unnamed_cols = [col for col in df.columns if 'Unnamed' in str(col)]
    if unnamed_cols:
        print(f"⚠ WARNING: Found {len(unnamed_cols)} unnamed columns: {unnamed_cols}")
    else:
        print("✓ All columns have names")

    # Check 4: Basic column overview
    print(f"\n✓ Dataset has {df.shape[1]} columns:")
    print(f"  - Column names: {list(df.columns)}")

## 1. ALF01.csv

In [52]:
df=load_data("ALF01.csv")

✓ Successfully loaded dataset with 5040 rows and 14 columns


,STATISTIC,Statistic Label,TLIST(A1),Year,C02076V02508,Age Group,C02199V02655,Sex,C04283V05060,Education Attainment Level,C03788V04538,NUTS 2 Region,UNIT,VALUE
0,ALF01C01,Annual employment rate,2019,2019,-,All ages,-,Both sexes,-,Levels of Education (Levels 0-8),IE0,Ireland,%,74.9
1,ALF01C01,Annual employment rate,2019,2019,-,All ages,-,Both sexes,-,Levels of Education (Levels 0-8),811D29A4-1E05-4F4A-B61B-43DE64E07B29,Northern and Western,%,74.0
2,ALF01C01,Annual employment rate,2019,2019,-,All ages,-,Both sexes,-,Levels of Education (Levels 0-8),97227605-23D5-4B98-BDB8-3EB5D0682A5A,Southern,%,72.8
3,ALF01C01,Annual employment rate,2019,2019,-,All ages,-,Both sexes,-,Levels of Education (Levels 0-8),C32761EB-7A21-4B95-8C7A-DB195C2955CE,Eastern and Midland,%,76.6
4,ALF01C01,Annual employment rate,2019,2019,-,All ages,-,Both sexes,10,Less than primary (Level 0),IE0,Ireland,%,NaN


In [53]:
validate_data(df)

Dataset Validation Checks:
✓ Dataset contains 5,040 rows
✓ No duplicate rows found
✓ All columns have names

✓ Dataset has 14 columns:
  - Column names: ['STATISTIC', 'Statistic Label', 'TLIST(A1)', 'Year', 'C02076V02508', 'Age Group', 'C02199V02655', 'Sex', 'C04283V05060', 'Education Attainment Level', 'C03788V04538', 'NUTS 2 Region', 'UNIT', 'VALUE']


In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5040 entries, 0 to 5039
Data columns (total 14 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   STATISTIC                   5040 non-null   object 
 1   Statistic Label             5040 non-null   object 
 2   TLIST(A1)                   5040 non-null   int64  
 3   Year                        5040 non-null   int64  
 4   C02076V02508                5040 non-null   object 
 5   Age Group                   5040 non-null   object 
 6   C02199V02655                5040 non-null   object 
 7   Sex                         5040 non-null   object 
 8   C04283V05060                5040 non-null   object 
 9   Education Attainment Level  5040 non-null   object 
 10  C03788V04538                5040 non-null   object 
 11  NUTS 2 Region               5040 non-null   object 
 12  UNIT                        5040 non-null   object 
 13  VALUE                       3037 

In [55]:


# Convert all object columns to categorical
for col in df.select_dtypes('object').columns:
    df[col] = df[col].astype('category')

df['Year']=df['Year'].astype('category')
df['TLIST(A1)']=df['TLIST(A1)'].astype('category')

# Update column groups after type corrections
numeric_columns = df.select_dtypes(['int64', 'float64']).columns
category_columns = df.select_dtypes('category').columns

print("✓ Data types corrected:")
print(f"  - Numeric features: {len(numeric_columns)}")
print(f"  - Categorical features: {len(category_columns)}")

✓ Data types corrected:
  - Numeric features: 1
  - Categorical features: 13


In [56]:
df_numeric_dqr = build_dqr_table(df, numeric_columns)
display(df_numeric_dqr)

,count,mean,std,min,25%,50%,75%,max,%missing,cardinality
VALUE,3037.0,74.34623,13.911852,24.3,66.0,76.9,85.2,100.0,39.742063,578


In [57]:
df_categorical_dqr = build_dqr_table(df, category_columns)
display(df_categorical_dqr)

,count,unique,top,freq,%missing,cardinality
STATISTIC,5040,1,ALF01C01,5040,0.0,1
Statistic Label,5040,1,Annual employment rate,5040,0.0,1
TLIST(A1),5040,7,2019,720,0.0,7
Year,5040,7,2019,720,0.0,7
C02076V02508,5040,10,-,504,0.0,10
Age Group,5040,10,20 - 24 years,504,0.0,10
C02199V02655,5040,3,-,1680,0.0,3
Sex,5040,3,Both sexes,1680,0.0,3
C04283V05060,5040,6,-,840,0.0,6
Education Attainment Level,5040,6,Less than primary (Level 0),840,0.0,6


In [58]:
# Count each value in every categorical column, including missing values.
for col in category_columns:
    counts = (
        df[col]
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="count")
    )

    print(f"\n{col}")
    display(counts)


STATISTIC


,value,count
0,ALF01C01,5040



Statistic Label


,value,count
0,Annual employment rate,5040



TLIST(A1)


...

,value,count
0,811D29A4-1E05-4F4A-B61B-43DE64E07B29,1260
1,97227605-23D5-4B98-BDB8-3EB5D0682A5A,1260
2,C32761EB-7A21-4B95-8C7A-DB195C2955CE,1260
3,IE0,1260



NUTS 2 Region


,value,count
0,Eastern and Midland,1260
1,Ireland,1260
2,Northern and Western,1260
3,Southern,1260



UNIT


,value,count
0,%,5040


In [59]:
for col in category_columns:
    missing_stats = (
        df.groupby(col, observed=True, dropna=False)["VALUE"]
        .agg(
            total_rows="size",
            valid_rows="count",
        )
    )

    missing_stats["missing_rows"] = (
        missing_stats["total_rows"] - missing_stats["valid_rows"]
    )
    missing_stats["missing_pct"] = (
        missing_stats["missing_rows"]
        .div(missing_stats["total_rows"])
        .mul(100)
        .round(2)
    )

    print(f"\n{col}")
    display(missing_stats.sort_values("missing_pct", ascending=False))


STATISTIC


,total_rows,valid_rows,missing_rows,missing_pct
STATISTIC,,,,
ALF01C01,5040,3037,2003,39.74



Statistic Label


,total_rows,valid_rows,missing_rows,missing_pct
Statistic Label,,,,
Annual employment rate,5040,3037,2003,39.74



TLIST(A1)


...

,total_rows,valid_rows,missing_rows,missing_pct
C03788V04538,,,,
811D29A4-1E05-4F4A-B61B-43DE64E07B29,1260,635,625,49.60
97227605-23D5-4B98-BDB8-3EB5D0682A5A,1260,760,500,39.68
C32761EB-7A21-4B95-8C7A-DB195C2955CE,1260,791,469,37.22
IE0,1260,851,409,32.46



NUTS 2 Region


,total_rows,valid_rows,missing_rows,missing_pct
NUTS 2 Region,,,,
Northern and Western,1260,635,625,49.60
Southern,1260,760,500,39.68
Eastern and Midland,1260,791,469,37.22
Ireland,1260,851,409,32.46



UNIT


,total_rows,valid_rows,missing_rows,missing_pct
UNIT,,,,
%,5040,3037,2003,39.74
